In [1]:
!pip install -U sentence-transformers rank_bm25

In [14]:
import gzip
import json
import os

import torch

from sentence_transformers.util import http_get

if not torch.cuda.is_available():
    print("Warning: No GPU found. Please add GPU to your notebook")

# As dataset, we use Simple English Wikipedia. Compared to the full English wikipedia, it has only
# about 170k articles. We split these articles into paragraphs.
wikipedia_filepath = "simplewiki-2020-11-01.jsonl.gz"

if not os.path.exists(wikipedia_filepath):
    http_get("http://sbert.net/datasets/simplewiki-2020-11-01.jsonl.gz", wikipedia_filepath)

passages = []
with gzip.open(wikipedia_filepath, "rt", encoding="utf8") as fIn:
    for line in fIn:
        data = json.loads(line.strip())

        # Add all paragraphs
        # passages.extend(data['paragraphs'])

        # Only add the first paragraph
        passages.append(data["paragraphs"][0])

print("Passages:", len(passages))

passages = passages[:10000]

Passages: 169597


In [ ]:
import string

import numpy as np
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction import _stop_words
from tqdm.autonotebook import tqdm


def bm25_tokenizer(text):
    tokenized_doc = []
    for token in text.lower().split():
        token = token.strip(string.punctuation)

        if len(token) > 0 and token not in _stop_words.ENGLISH_STOP_WORDS:
            tokenized_doc.append(token)
    return tokenized_doc


tokenized_corpus = []
for passage in tqdm(passages):
    tokenized_corpus.append(bm25_tokenizer(passage))

bm25 = BM25Okapi(tokenized_corpus)      # default k1=1.5, b=0.75, epsilon=0.25

  0%|          | 0/10000 [00:00<?, ?it/s]

In [16]:
query = "Number countries Europe"
bm25_scores = bm25.get_scores(bm25_tokenizer(query))
len(bm25_scores)

10000

In [17]:
def search_bm25(query, bm25, top_k=3):
    bm25_scores = bm25.get_scores(bm25_tokenizer(query))
    top_n = np.argpartition(bm25_scores, -top_k-2)[-top_k-2:]
    bm25_hits = [{"corpus_id": idx, "score": bm25_scores[idx]} for idx in top_n]
    bm25_hits = sorted(bm25_hits, key=lambda x: x["score"], reverse=True)

    print("Top-3 lexical search (BM25) hits")
    for hit in bm25_hits[0:3]:
        print("\t{:.3f}\t{}".format(hit["score"], passages[hit["corpus_id"]]))

In [18]:
search_bm25(query, bm25)

Top-3 lexical search (BM25) hits
	10.661	Equestrianism is the sport of horseback riding. It is a popular sport in countries like the United States, Australia, the United Kingdom and other countries in Europe. Horses are used in many different competitions.
	10.486	The Baltic Sea is a sea in northern Europe between Scandinavia, Finland, Russia, the Baltic countries, Poland, and Germany.
	10.024	The telephone number 1-1-2 (or 112) is the standard European Union (EU) emergency telephone number, it works in every country of the EU, for land lines and mobile phones. It is also used in some other countries as an emergency telephone number for both mobile and fixed-line telephones.
